## Using ResNet (Transfer Learning) Step by Step

Instead of training from scratch, we’ll use a pretrained model:

### What is ResNet?

ResNet = Residual Network

It was introduced in:

Microsoft Research

Paper: “Deep Residual Learning for Image Recognition”

Authors include Kaiming He

Popular model:

ResNet (we'll use ResNet18)

### Why ResNet is Special 

In normal deep networks:

H(x) ,is what the layer tries to learn.

In ResNet, the layer learns:
F(x)=H(x)−x

So the output becomes:

H(x)=F(x)+x

This is called a skip connection or residual connection.

Instead of learning full mapping, it learns the difference.

Why is this powerful?

Because deep networks suffer from:

Vanishing gradients which occurs when the gradient is small, it causes problem in backpropagation.

Degradation problem (accuracy gets worse when network gets deeper)

Residual connections help gradients flow directly backward.

## ResNet18 Transfer Learning

### STEP 1 — Imports

In [27]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
import os

### STEP 2 — Device

In [28]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


### STEP 3 — Dataset Path

In [29]:
path = "C:/Users/OLIVE/.cache/kagglehub/datasets/mayank07thakur/happy-vs-sad-people-cnn-classification/versions/1"


### STEP 4 — Transforms
IMPORTANT: ResNet needs ImageNet normalization

In [30]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),        # ResNet default input size
    transforms.ToTensor(),
    transforms.Normalize(                   # Normalize images using ImageNet mean and std, which is important when using pre-trained models like ResNet that were trained on ImageNet; This ensures that the input images are normalized in the same way as the images used to train the model, which can help improve performance
        mean=[0.485, 0.456, 0.406],         # ImageNet mean
        std=[0.229, 0.224, 0.225]           # ImageNet std
    )                                       
])


### STEP 5 — Load Dataset

In [31]:
full_dataset = datasets.ImageFolder(root=path, transform=transform)

print("Classes:", full_dataset.classes)
print("Total images:", len(full_dataset))

Classes: ['Happy People', 'Sad People']
Total images: 351


### STEP 6 — Train/Validation Split

In [32]:
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

print("Train size:", len(train_dataset))
print("Validation size:", len(val_dataset))

Train size: 280
Validation size: 71


### STEP 7 — DataLoaders

In [33]:
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

### STEP 8 — Load Pretrained ResNet18

In [34]:
model = models.resnet18(pretrained=True)

### Freeze all layers

In [35]:
for param in model.parameters():              # freeze all model parameters
    param.requires_grad = False               # freeze all model parameters so that they are not updated during training; This is important when using a pre-trained model for transfer learning, as we typically want to keep the learned features from the pre-trained model and only train the final classification layer on our specific task

### Replace final layer

In [36]:
num_features = model.fc.in_features             # Get the number of input features to the final fully connected layer of ResNet, which is necessary to replace it with a new layer that has the correct number of input features for our specific classification task
model.fc = nn.Linear(num_features, 2)           # Replace the final fully connected layer of ResNet with a new layer that has 2 output features (for happy and sad classes), with the same number of input features as the original layer; This allows us to use the pre-trained ResNet model for our specific classification task by only training the final layer while keeping the learned features from the pre-trained model intact

### Only final layer trainable

In [37]:
for param in model.fc.parameters():           # unfreeze the parameters of the final fully connected layer so that they can be updated during training; This is necessary because we want to train the final layer on our specific classification task while keeping the rest of the pre-trained model frozen
    param.requires_grad = True

model = model.to(device)               # Move the model to the appropriate device (GPU if available, otherwise CPU)

### STEP 9 — Loss and Optimizer

In [38]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)     # Only optimize the parameters of the final fully connected layer since the rest of the model is frozen

### STEP 10 — Training Loop

In [39]:
torch.cuda.empty_cache()             # Clear the GPU cache to free up memory before starting training; This can help prevent out-of-memory errors when training on a GPU, especially if there are other processes using the GPU or if the model and data are large

epochs = 15

for epoch in range(epochs):
    model.train()
    running_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_acc = 100 * correct / total

    print(f"Epoch [{epoch+1}/{epochs}] "
          f"Loss: {running_loss:.3f} "
          f"Train Accuracy: {train_acc:.2f}%")

Epoch [1/15] Loss: 10.009 Train Accuracy: 70.00%
Epoch [2/15] Loss: 6.532 Train Accuracy: 88.93%
Epoch [3/15] Loss: 5.622 Train Accuracy: 89.29%
Epoch [4/15] Loss: 5.245 Train Accuracy: 88.21%
Epoch [5/15] Loss: 4.444 Train Accuracy: 89.29%
Epoch [6/15] Loss: 5.380 Train Accuracy: 88.93%
Epoch [7/15] Loss: 5.047 Train Accuracy: 88.21%
Epoch [8/15] Loss: 4.387 Train Accuracy: 88.93%
Epoch [9/15] Loss: 3.538 Train Accuracy: 92.86%
Epoch [10/15] Loss: 3.600 Train Accuracy: 93.57%
Epoch [11/15] Loss: 3.676 Train Accuracy: 92.50%
Epoch [12/15] Loss: 2.705 Train Accuracy: 96.43%
Epoch [13/15] Loss: 4.232 Train Accuracy: 88.93%
Epoch [14/15] Loss: 3.088 Train Accuracy: 93.21%
Epoch [15/15] Loss: 2.730 Train Accuracy: 95.00%


### STEP 11 — Validation

In [40]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

val_acc = 100 * correct / total

print("Validation Accuracy:", val_acc)

Validation Accuracy: 87.32394366197182
